In [4]:
import pandas as pd
import os, uuid

In [ ]:
data_dir = os.path.join(os.getcwd(), "esco_cs/")
out_dir = os.path.join(os.getcwd(), "formatted/")
os.makedirs(out_dir, exist_ok=True)

### merge CSV files containing specific skill additions into one file

In [ ]:
combined = pd.concat(
    [pd.read_csv(f) for f in os.listdir('esco_cs')
     # removed transversalSkillsCollection_cs due to how many less relevant and more too-specific records it added
     ],
    ignore_index=True
)

combined.to_csv(f'{out_dir}combinedSkills_cs.csv', index=False)
combined_occ = pd.concat(
    [pd.read_csv(f'{data_dir}occupations_cs.csv'),
     pd.read_csv(f'{data_dir}researchOccupationsCollection_cs.csv')],
    ignore_index=True
)
combined_occ.to_csv(f'{out_dir}combinedOccupations_cs.csv',index=False)

### Util functions
- replace URLs used in ESCO datasets with UUIDs
- format incosistently split skill altlabels (eg pipe, tab, newline) and store as list
- enables category lookup using `formatted/occupation_hierarchy.csv` | `formatted/skill_hierarchy.csv`

In [2]:
uri_to_uuid_map = {}

def store_uri(uri):
    key = f'key_{len(uri_to_uuid_map) + 1}'
    uri_to_uuid_map[uri.strip().lower()] = key
    return key

def get_uuid_from_concept_uri(concept_uri):
    return uri_to_uuid_map.get(concept_uri.strip().lower())

def format_alt_labels(alts_str):
    a = (
    alts_str
        .replace('\xa0', ' ')
        .replace('|', '\n')
        .replace('0.0', '')
    )
    alts = '\n'.join(aa.strip() for aa in a.splitlines())

    return '' if alts == 'nan' else alts

### define record formats (taken from tabiya transform.js)
= formats of data to work with without extra relations

_ISCO == international skill groups (o*net)_

In [3]:
def isco_groups_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": record["altLabels"], # currently no alt labeling exists in the czech dataset,so no need to format here for now
        "DESCRIPTION": record["description"],
    }


def skills_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "SKILLTYPE": record.get("skillType"),
        "REUSELEVEL": record.get("reuseLevel"),
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": format_alt_labels(str(record["altLabels"])),
        "DESCRIPTION": record["description"],
        "DEFINITION": record.get("definition"),
        "SCOPENOTE": record.get("scopeNote"),
    }


def occupations_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record["conceptUri"]),
        "UUIDHISTORY": str(uuid.uuid4()),
        "ISCOGROUPCODE": record["iscoGroup"],
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": format_alt_labels(str(record["altLabels"])),
        "DESCRIPTION": record["description"],
        "DEFINITION": record["definition"],
        "SCOPENOTE": record.get("scopeNote"),
        "REGULATEDPROFESSIONNOTE": record.get("regulatedProfessionNote"),
        "OCCUPATIONTYPE": "ESCO",
    }


def skill_groups_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": record["altLabels"],
        "DESCRIPTION": record["description"],
        "SCOPENOTE": record.get("scopeNote"),
    }


def skills_skills_relations_transformer(record):
    requiring_id = get_uuid_from_concept_uri(record['originalSkillUri'])
    required_id = get_uuid_from_concept_uri(record['relatedSkillUri'])

    return {
        "REQUIRINGID": requiring_id,
        "RELATIONTYPE": record['relationType'],
        "REQUIREDID": required_id
    }


def occupations_skills_relations_transformer(record):
    occupation_id = get_uuid_from_concept_uri(record['occupationUri'])
    skill_id = get_uuid_from_concept_uri(record['skillUri'])
    occupation_type = record.get('occupationType', "ESCO")

    return {
        "OCCUPATIONTYPE": occupation_type,
        "OCCUPATIONID": occupation_id,
        "RELATIONTYPE": record['relationType'],
        "SKILLID": skill_id
    }


def occupation_skill_hierarchy_transformer(record):
    parent_id = get_uuid_from_concept_uri(record['broaderUri'])
    child_id = get_uuid_from_concept_uri(record['conceptUri'])

    return {
        "PARENTOBJECTTYPE": record['broaderType'],
        "PARENTID": parent_id,
        "CHILDID": child_id,
        "CHILDOBJECTTYPE": record['conceptType']
    }


### Reformat

In [4]:

def export_all(filepath, outputs, templates):
    for filename, record in templates:
        inputf = os.path.join(filepath, filename)
        output = os.path.join(outputs, filename)
        df = pd.read_csv(inputf)
        transformed = df.apply(record, axis=1)
        transformed = pd.DataFrame(transformed.tolist())
        transformed.to_csv(output, index=False)


template_map = [
    ('ISCOGroups_cs.csv', isco_groups_transformer),
    ('combinedOccupations_cs.csv', occupations_transformer),
    ('combinedSkills_cs.csv', skills_transformer),
    ('skillGroups_cs.csv', skill_groups_transformer),
    ('broaderRelationsOccPillar_cs.csv', occupation_skill_hierarchy_transformer),
    ('broaderRelationsSkillPillar_cs.csv', occupation_skill_hierarchy_transformer),
    ('occupationSkillRelations_cs.csv', occupations_skills_relations_transformer),
    ('skillSkillRelations_cs.csv', skills_skills_relations_transformer)
]

In [5]:
export_all(data_dir, out_dir, template_map)

#### deduplicate skills

In [5]:
df = pd.read_csv(f'{data_dir}combinedSkills_cs.csv', encoding='utf-8')
seenl = {}
for idx,r in df.iterrows():
    l = r.get('PREFERREDLABEL','')
    desc = r.get('DESCRIPTION','')
    if l in seenl and seenl[l] == desc:
        df.drop(idx, inplace=True)
    else: seenl[l] = desc

df.to_csv(f'formatted/combinedSkills_cs_cleaned.csv', encoding='utf-8')

#### split altlabels so each has its own row

In [ ]:
origin = pd.read_csv(f'{data_dir}combinedOccupations_cs.csv')

def denull(row):
    if row['ALTLABEL']=='':
        row['ALTLABEL'] = row['PREFERREDLABEL']
    return row

split_titles = origin.copy()
split_titles['ALTLABEL'] = (
    split_titles['ALTLABELS']
        .fillna('')
        .str.split('\n')
)
split_titles = split_titles.explode('ALTLABEL')
split_titles['ALTLABEL'] = split_titles['ALTLABEL'].str.strip()
split_titles = split_titles.apply(denull,axis=1)
split_titles.to_csv(f'{out_dir}combinedOccupations_augmented_cs.csv')

#### merge resume files into jsonl for eval

In [3]:
lines = []
for i in range(1,571):
    with (open(f'resumes/{i}.txt','r') as r):
        text = ' '.join(
            r.read()
            .replace('\xa0', '|')
            .split())
        lines.append(text)

with open("formatted/resumes.json", 'w') as f:
    f.write(json.dumps(lines, indent=4, ensure_ascii=False))